# Deep Learning - based Network Analysis

## Configuration

In [1]:
import pandas as pd
import os

import plotly.express as px

In [2]:
pickle_path = "../data/UniProtNormalizedTabular-default.txt"
dl_net_path = "../data/dl_net.txt"

## Preliminaries (Correct?)

### Pickle Database

In [3]:
pickle_df = pd.read_csv(pickle_path, sep="\t", index_col=None)

# New dataframe for pickle proteins with 2 columns: Protein and Degree
pickle_degree_df = pd.concat([pickle_df["InteractorA"], pickle_df["InteractorB"]]).value_counts().rename_axis("Protein").reset_index(name="Degree")
pickle_degree_df

,Protein,Degree
0,P05067,2382
1,P42858,902
2,P04637,851
3,P0CG48,833
4,O15294,760
...,...,...
16415,P78411,1
16416,Q6PIV7,1
16417,P01242,1
16418,P25021,1


## Deep Learning - based prediction Network

In [4]:
dl_net_df = pd.read_csv(dl_net_path, sep="\t", index_col=None, header=None, names=["InteractorA", "InteractorB", "Score"])  
dl_net_df

,InteractorA,InteractorB,Score
0,A0A6Q8PGC7,D4JZ56,0.999603
1,S4R2Y5,D4JZ56,0.999023
2,E9PRC8,A0A0J6C3N6,0.999356
3,E9PRC8,A0A0J6C2H7,0.993397
4,E9PRC8,A0A0J6C3Q8,0.995980
...,...,...,...
16106272,E5RGN6,A0A1H4CMV9,0.995117
16106273,E5RGN6,A0A2K9P088,0.995605
16106274,E5RGN6,A0A413RC28,0.999512
16106275,E5RGN6,D4JZ56,1.000000


In [5]:
# Take all human proteins (InteractorA) and calculate their degree in the dl_net_df
dl_net_human_proteins = dl_net_df["InteractorA"].value_counts().rename_axis("Protein").reset_index(name="Degree")
dl_net_human_proteins

,Protein,Degree
0,E7ERC4,82893
1,P06748,71541
2,Q9Y252,68752
3,Q01844,68280
4,S4R2Z2,66662
...,...,...
19049,F8VPF2,1
19050,C9IZL6,1
19051,E9PG39,1
19052,P14868,1


In [6]:
def pickla_comparison(net_proteins, pickle_degree_df):
    '''
    Compare the degree of proteins in an input network (net_proteins) with the
    degree of the same proteins in the pickle (PICKLE) interaction database.
    '''
    # Normalize the protein id and degree columns of the input network
    net = net_proteins.copy()
    protein_col = [c for c in ["Protein", "Uniprot", "UniProt", "UniprotID", "UniProtID"] if c in net.columns]
    protein_col = protein_col[0] if protein_col else net.columns[0]
    degree_col = [c for c in ["Degree", "degree"] if c in net.columns]
    degree_col = degree_col[0] if degree_col else net.columns[1]
    net = net.rename(columns={protein_col: "Protein", degree_col: "degree"})

    # Use the degree of each protein in the pickle database
    pickle_degree = pickle_degree_df.rename(columns={"Degree": "pickle_degree"})

    # Merge dataframes on the UniProt id
    prot_degreeDF = pd.merge(net[["Protein", "degree"]], pickle_degree, on="Protein", how="inner")

    # Pearson correlation
    corr = prot_degreeDF[["degree", "pickle_degree"]].corr(method="pearson")
    print("Pearson correlation of degree and pickle_degree: ")
    print(corr)

    # Scatter plot
    fig = px.scatter(
        prot_degreeDF,
        x="degree",
        y="pickle_degree",
        hover_data=["Protein"]
    )

    # Update layout
    fig.update_layout(
        template="plotly_white",
        xaxis=dict(
            title=dict(
                text="Degree",
                font=dict(
                    family="Times New Roman",
                    size=50
                )
            ),
            tickfont=dict(
                family="Times New Roman",
                size=30
            ),
        ),
        yaxis=dict(
            title=dict(
                text="Pickle Degree",
                font=dict(
                    family="Times New Roman",
                    size=50
                )
            ),
            tickfont=dict(
                family="Times New Roman",
                size=30
            ),
        )
    )

    # Save the image
    img_name = "pickla_comparison.svg"
    fig.write_image(img_name, width=1920, height=1080, scale=1)


In [7]:
pickla_comparison(dl_net_human_proteins, pickle_degree_df)

Pearson correlation of degree and pickle_degree: 
                degree  pickle_degree
degree         1.00000        0.09673
pickle_degree  0.09673        1.00000
